##Transform Sprints Data

1. Read bronze sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId->constructor_id, driverId->driver_id, raceName->race_name, positionText->finish_position_text)
4. Rename columns to make them more meaningful (date->race_date, grid->grid_position, laps->completed_laps, number->car_number, position->finish_position)
5. Filter out rows where season, round, constructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns race_name to Title Case
8. Write the transformed data to silver sprints table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

####Step 1 to 7: Read, Transform, perform quality checks

In [0]:
sprints_df = (
    spark.table(bronze_table)
        .filter(F.col("batch_id") == v_batch_id)
        .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingestion_timestamp",
                "source_file",
                "batch_id"
            )
        .withColumnsRenamed({
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "driver_number",
                "position": "final_position",
                "positionText": "final_position_text"})
        .filter(
                F.col("season").isNotNull() &
                F.col("round").isNotNull() &
                F.col("constructor_id").isNotNull() &
                F.col("driver_id").isNotNull()
            )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
        .withColumn("race_name", F.initcap(F.col("race_name")))

)

####Sometimes the fully chained method becomes overwhelming and hence developers use a middle-ground approach where they only chain similar steps like selection + renaming, filtering + removing duplicates, etc

####Step 8: Write the transformed data to the silver 'sprints' table

In [0]:
display(sprints_df)

season,round,constructor_id,driver_id,race_date,race_name,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file,batch_id
2023,4,red_bull,perez,2023-04-30,Azerbaijan Grand Prix,2,17,11,8.0,1,1,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,ferrari,leclerc,2023-04-30,Azerbaijan Grand Prix,1,17,16,7.0,2,2,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,red_bull,max_verstappen,2023-04-30,Azerbaijan Grand Prix,3,17,1,6.0,3,3,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,mercedes,russell,2023-04-30,Azerbaijan Grand Prix,4,17,63,5.0,4,4,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,ferrari,sainz,2023-04-30,Azerbaijan Grand Prix,5,17,55,4.0,5,5,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,aston_martin,alonso,2023-04-30,Azerbaijan Grand Prix,8,17,14,3.0,6,6,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,mercedes,hamilton,2023-04-30,Azerbaijan Grand Prix,6,17,44,2.0,7,7,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,aston_martin,stroll,2023-04-30,Azerbaijan Grand Prix,9,17,18,1.0,8,8,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,williams,albon,2023-04-30,Azerbaijan Grand Prix,7,17,23,0.0,9,9,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01
2023,4,mclaren,piastri,2023-04-30,Azerbaijan Grand Prix,11,17,81,0.0,10,10,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01


In [0]:
write_to_silver(
    input_df = sprints_df,
    target_table = silver_table,
    merge_condition = "t.season=s.season AND t.round=s.round AND t.constructor_id=s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update = [
        "season",
        "round",
        "constructor_id",
        "driver_id",
        "race_date",
        "race_name",
        "grid_position",
        "completed_laps",
        "driver_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))

season,round,constructor_id,driver_id,race_date,race_name,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
2023,4,red_bull,max_verstappen,2023-04-30,Azerbaijan Grand Prix,3,17,1,6.0,3,3,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,4,williams,albon,2023-04-30,Azerbaijan Grand Prix,7,17,23,0.0,9,9,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,4,alfa,zhou,2023-04-30,Azerbaijan Grand Prix,14,17,24,0.0,12,12,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,4,alphatauri,tsunoda,2023-04-30,Azerbaijan Grand Prix,16,2,22,0.0,19,R,Collision damage,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,9,ferrari,leclerc,2023-07-02,Austrian Grand Prix,9,24,16,0.0,12,12,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,12,aston_martin,stroll,2023-07-30,Belgian Grand Prix,14,11,18,0.0,11,11,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,18,haas,hulkenberg,2023-10-22,United States Grand Prix,16,19,27,0.0,15,15,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2023,20,red_bull,max_verstappen,2023-11-05,São Paulo Grand Prix,2,24,1,8.0,1,1,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2023.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2024,6,mclaren,piastri,2024-05-05,Miami Grand Prix,6,19,81,3.0,6,6,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2024.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
2024,6,haas,kevin_magnussen,2024-05-05,Miami Grand Prix,14,19,20,0.0,18,18,Finished,2026-08-02T16:49:41.980Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/sprints/sprints_2024.json,2025-01,2026-08-05T15:24:39.756Z,2026-08-05T15:24:53.556Z
